In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
from go_ml.eval_utils import filter_annot_df
from go_ml.eval_utils import (load_msa_dict, gen_bert_mat, get_bert_entropy, 
                              gen_annot_mat, gen_seq_len_mask, mean_reciprocal_rank, 
                              mean_reciprocal_rank_mat, mean_auc, top_30_score, roc_average)

data_root = '../gen_datasets/datasets'
csa_df = filter_annot_df(pd.read_csv(f'{data_root}/csa_annot.csv', sep='\t'))
llps_df = filter_annot_df(pd.read_csv(f'{data_root}/llps_dataset.csv', sep='\t'))
elms_df = filter_annot_df(pd.read_csv(f'{data_root}/elms_dataset.csv', sep='\t'))

In [2]:
len(csa_df), len(llps_df), len(elms_df)

(784, 53, 229)

In [9]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
# from esm.models.esmc import ESMC
import torch
from transformers import AutoModelForMaskedLM, AutoTokenizer

device = torch.device('cuda:0')
# model_path = 'Synthyra/FastESM2_650'
# model_path = 'facebook/esm2_t33_650M_UR50D'
model_path = 'Synthyra/ESM2-650M'
model = AutoModelForMaskedLM.from_pretrained(model_path, torch_dtype=torch.float16, trust_remote_code=True).eval().to(device)
#Check if model has tokenizer attribute
if hasattr(model, 'tokenizer'):
    tokenizer = model.tokenizer
else:
    tokenizer = AutoTokenizer.from_pretrained(model_path, use_fast=True)
# tokenizer = model.tokenizer
vi = {i: a for a, i in tokenizer.get_vocab().items()}
AA_str = [vi[i] for i in range(4, 24)]
from go_ml.masking import mask_perc, get_logits_esmfast

config.json: 0.00B [00:00, ?B/s]

modeling_fastesm.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/Synthyra/ESM2-650M:
- modeling_fastesm.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors:   0%|          | 0.00/2.60G [00:00<?, ?B/s]

In [10]:
from tqdm.notebook import tqdm
import pickle

ds_labels = ['csa', 'llps', 'elms']
ds_l = [csa_df, llps_df, elms_df]
for ds_label, annot_df in zip(ds_labels, ds_l):
    df_logits = {}
    for i, (seq_id, seq) in tqdm(enumerate(zip(annot_df['UniprotID'], annot_df['Sequence'])), 
                                 total=len(annot_df), desc=f'Processing {ds_label}'):
        df_logits[seq_id] = get_logits_esmfast(seq, model, tokenizer, batch_size=8, mask_func=mask_perc).float()
    bert_map = {k: v[:, 4:24] for k, v in df_logits.items()}
    bert_map = {k: v / (v.sum(dim=1, keepdims=True) + 1e-10) for k, v in bert_map.items()}
    bert_map = {k: v.numpy() for k, v in bert_map.items()}

    annot_mat = gen_annot_mat(annot_df['AnnotatedIndices'], [len(s) for s in annot_df['Sequence']])
    seq_len_mask = gen_seq_len_mask(annot_df['Sequence'])
    bert_mat = gen_bert_mat(annot_df['UniprotID'], bert_map, max_len=850)
    bert_entropy = get_bert_entropy(bert_mat, seq_len_mask)
    with open(f'eval_files/{ds_label}_esm_fast_esm2.pkl', 'wb') as f:
            pickle.dump({'UniprotID': annot_df['UniprotID'], 'bert_mat': bert_mat, 
                        'seq_len_mask': seq_len_mask, 'bert_entropy': bert_entropy}, f)
    print(f'DS label: {ds_label}')

Processing csa:   0%|          | 0/784 [00:00<?, ?it/s]

















































































































































































































































































































































































































































































































































































































































































































































































































DS label: csa


Processing llps:   0%|          | 0/53 [00:00<?, ?it/s]






















































DS label: llps


Processing elms:   0%|          | 0/229 [00:00<?, ?it/s]






































































































































































































































DS label: elms


In [14]:
test_seq = 'MKTAYIAKQRQISFVKSHFSRQDILDLWIYHTQGYFPDWQNYTPGPGIRYPLKFRTLSVPCGYIAGGTFDVSIL'
seq_data = tokenizer.batch_encode_plus(
            [test_seq, test_seq+'AA'],
            add_special_tokens=True,
            padding='longest',
            truncation=True,
            max_length=1024,
            return_tensors='pt'
        )
seq_ind = seq_data['input_ids']
mask = seq_data['attention_mask']
print(seq_ind.shape, mask.shape)

torch.Size([2, 78]) torch.Size([2, 78])


32